# Resell Copilot — Qwen v3 field-eval

Re-runs the field-level evaluation of `mchlkan/qwen3vl4b-resell-adapter-multi-v3` on the locked 500-row test split.
Results land in `eval/results/qwen_field_eval.json` and overwrite the stale v1 numbers from 2026-05-08.

**Before running:** Runtime → Change runtime type → **GPU** (free T4 is enough with `--load-in-4bit`).

Total time: ~30 min (T4) or ~15 min (A100/L4). The script checkpoints every 25 rows, so a disconnected runtime can be resumed by re-running the same cell.


In [1]:
# 1. GPU check
!nvidia-smi


Wed May 13 20:54:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Clone the repo
!git clone https://github.com/mchlkan/Advanced_ML.git
%cd Advanced_ML
!git checkout feature/deploy-prep


Cloning into 'Advanced_ML'...
remote: Enumerating objects: 1403, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 1403 (delta 20), reused 28 (delta 14), pack-reused 1359 (from 2)
Receiving objects: 100% (1403/1403), 10.20 MiB | 27.35 MiB/s, done.
Resolving deltas: 100% (865/865), done.
/content/Advanced_ML
Branch 'feature/deploy-prep' set up to track remote branch 'feature/deploy-prep' from 'origin'.
Switched to a new branch 'feature/deploy-prep'


In [3]:
# 3. Install dependencies (~2-3 min on a fresh runtime)
!pip install -q -U torch transformers peft accelerate bitsandbytes \
                   pandas pyarrow pillow tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 7.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 71.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━

## Data files

The script reads three things from `data/`:

- `vinted_clothing_combined.parquet`
- `kleinanzeigen_clothing_combined.parquet`
- `splits/test_ids.json` (and optionally `splits/test.parquet`)

They're gitignored, so you need to ship them into the Colab session. **Pick the option that matches where you keep them today** (only run one of the next two cells).


In [ ]:
# Option A — pull from Google Drive (fastest if you keep a copy there)
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
SRC = '/content/drive/MyDrive/resell_copilot'   # ← edit this if your folder lives elsewhere

os.makedirs('data/splits', exist_ok=True)
for fn in ['vinted_clothing_v2_en.parquet', 'kleinanzeigen_clothing_v1_en.parquet']:
    shutil.copy(f'{SRC}/{fn}', f'data/{fn}')
for fn in os.listdir(f'{SRC}/splits'):
    if fn.startswith('test'):
        shutil.copy(f'{SRC}/splits/{fn}', f'data/splits/{fn}')

!ls -la data/ data/splits/


Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/resell_copilot/vinted_clothing_combined.parquet'

In [ ]:
# Option B — upload manually via the Files panel on the left, then verify here
# (drag the three files into data/ and data/splits/ in the Colab file browser)
!ls -la data/ data/splits/ 2>/dev/null || echo 'data/ does not exist yet — upload the files first'


In [ ]:
# 4. Sanity check — confirm the script's expected inputs are in place
import pathlib
required = [
    'data/vinted_clothing_combined.parquet',
    'data/kleinanzeigen_clothing_combined.parquet',
    'data/splits/test_ids.json',
]
missing = [p for p in required if not pathlib.Path(p).exists()]
for p in required:
    print(f'{p:60s}  {"OK" if pathlib.Path(p).exists() else "MISSING"}')
if missing:
    raise SystemExit(f'\nMissing files: {missing}. Re-run the data-copy cell.')


## (Optional) HuggingFace token

The v3 adapter is public, so `HF_TOKEN` is only needed if you hit a rate limit during the base-model download. Uncomment and set if needed.


In [ ]:
# import os
# os.environ['HF_TOKEN'] = 'hf_...'


## Smoke test (5 rows, ~2 min)

If this succeeds, the full benchmark will too. Confirms the script imports cleanly, the model loads on this GPU, and the prompt schema parses end-to-end.


In [ ]:
!python eval/run_qwen_field_eval.py --load-in-4bit --limit 5 \
  --predictions /tmp/qwen_pred_smoke.parquet \
  --metrics /tmp/qwen_eval_smoke.json

import json
with open('/tmp/qwen_eval_smoke.json') as f:
    print(json.dumps(json.load(f), indent=2))


## Full 500-row benchmark (~20–30 min on T4, ~10–15 min on A100/L4)

Output overwrites `eval/results/qwen_field_eval.json` and `eval/results/qwen_field_predictions.parquet`. If the runtime disconnects, just re-run this cell — the per-row predictions parquet is checkpointed every 25 rows and will resume.


In [ ]:
!python eval/run_qwen_field_eval.py --load-in-4bit


In [ ]:
# 5. Inspect the result file and pull out the headline numbers for the slide
import json
with open('eval/results/qwen_field_eval.json') as f:
    metrics = json.load(f)

print('=== full JSON ===')
print(json.dumps(metrics, indent=2))

print('\n=== headline numbers ===')
print(f'  adapter:           {metrics.get("adapter_id")}')
print(f'  total rows:        {metrics.get("total")}')
if metrics.get('parse_rate') is not None:
    print(f'  parse_rate:        {metrics["parse_rate"]:.1%}   (v1: 81.8%)')

acc = metrics.get('accuracy', {})
v1_baseline = {'brand': 0.475, 'category': 0.968, 'condition': 0.656, 'color': 0.722, 'size': 0.240}
for field in ('brand', 'category', 'condition', 'color', 'size'):
    a = acc.get(field, {})
    if a.get('exact') is not None:
        v1 = v1_baseline.get(field)
        delta = f'  (Δ {(a["exact"] - v1) * 100:+.1f} pp vs v1 {v1:.1%})' if v1 is not None else ''
        print(f'  {field:18s}  exact={a["exact"]:.1%}  (n={a.get("n")}){delta}')

fz = acc.get('brand', {}).get('fuzzy')
if fz is not None:
    print(f'  brand fuzzy:        {fz:.1%}')

p = metrics.get('vlm_price_eur', {})
if p.get('mape') is not None:
    print(f'  price MAPE:         {p["mape"]:.3f}   (vs GPT-4o-mini 1.025)')
    print(f'  price MAE:          €{p["mae"]:.2f}')


## Download the results

Run the cell below, **or** right-click each file in the Files panel → *Download*.


In [ ]:
from google.colab import files
files.download('eval/results/qwen_field_eval.json')
files.download('eval/results/qwen_field_predictions.parquet')


## (Optional) Back up to Google Drive


In [ ]:
import shutil
DEST = '/content/drive/MyDrive/resell_copilot'
shutil.copy('eval/results/qwen_field_eval.json', f'{DEST}/qwen_field_eval.json')
shutil.copy('eval/results/qwen_field_predictions.parquet', f'{DEST}/qwen_field_predictions.parquet')
print('backed up to', DEST)
